In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error
import numpy as np

import sys
import os
sys.path.append(os.path.abspath(".."))

from bblr.bblr2 import *
from bblr.lowess import *

In [ ]:
def prepare_and_split_data(path, cat_features, num_features, target, test_size):
    # 1. Upload dataset
    df = pd.read_csv(path)
    
    # 2. One-Hot Encoding and Standard Scaling 
    X = df.drop(columns=[target])
    y = df[target]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
        ])
    
    # Splitting the data into training and test sets
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # Scaling the data
    X_train = preprocessor.fit_transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)
    
    return X_train, X_test, y_train, y_test

def optimize_ball_based_regression(X_train, y_train, eps_range=np.arange(0.3, 0.8, 0.025), min_points_range=[10, 20, 30, 40, 50, 60, 70, 80]):
    # Defining parameter grid for GridSearchCV
    param_grid = {
        'eps': eps_range,
        'min_points': min_points_range,
        'coef_comp': ['mean', 'median']
    }
    
    # Initialising the model
    model = BallBasedLinearRegression()
    
    # Optimising hyperparameters with GridSearchCV
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search.best_params_

def optimize_lowess(X_train, y_train):
    param_grid = {
        'frac': np.arange(0.05, 0.4, 0.01)
    }
    
    lowess = LOWESSRegressor()
    
    grid_search = RandomizedSearchCV(
        estimator=lowess,
        param_distributions=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search.best_params_

In [3]:
file_path = "C:/Users/sylwester/PycharmProjects/bblr_testing/scripts/automl_datasets/boston.csv"
cat_features = []  # Assuming no categorical features in the Boston dataset
num_features = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
target = 'MEDV'
test_size = 0.2

In [4]:
X_train, X_test, y_train, y_test = prepare_and_split_data(file_path, cat_features, num_features, target, test_size)

In [5]:
# y_train = np.array(y_train)

In [40]:
best_est, best_par = optimize_ball_based_regression(X_train, y_train)

In [41]:
best_est

,eps,np.float64(0.3)
,column_names,[]
,min_points,40
,regression_type,'OLS'
,coef_comp,'mean'
,outliers_method,'up_to_N'
,eps_multiplier,0.1
,eps_range,"(0.1, ...)"
,ci,False
,pred_outliers_method,None
,confidence,0.95


In [23]:
best_par

{'coef_comp': 'mean', 'eps': np.float64(1.0999999999999999), 'min_points': 10}

In [42]:
bblr_best = best_est
bblr_best.fit(X_train, y_train)
y_pred_best = bblr_best.predict(X_test)
rmse_best = root_mean_squared_error(y_test, y_pred_best)
print(f"Default BBLR RMSE: {rmse_best}")

Default BBLR RMSE: 3.8120144543834917


In [36]:
bblr_default = BallBasedLinearRegression(eps=0.6, min_points=80, coef_comp='mean')
bblr_default.fit(X_train, y_train)
y_pred_default = bblr_default.predict(X_test)
rmse_default = root_mean_squared_error(y_test, y_pred_default)
print(f"Default BBLR RMSE: {rmse_default}")

Default BBLR RMSE: 3.874009212123776


In [8]:
# y_pred_train_default = bblr_default.predict(X_train)
# rmse_default = root_mean_squared_error(y_train, y_pred_train_default)
# print(f"Default BBLR RMSE: {rmse_default}")

In [46]:
param = 0.3
lowess = LOWESSRegressor(frac=0.3)
lowess.fit(X_train, y_train)
y_pred_lowess, coefs, intercepts, cis_raw, std_errors = lowess.predict(X_test, return_ci=False)
rmse_lowess = root_mean_squared_error(y_test, y_pred_lowess)
print(f"Default LOWESS RMSE: {rmse_lowess}")

Default LOWESS RMSE: 3.4431423827836287


In [5]:
best_est, best_par = optimize_lowess(X_train, y_train)

In [7]:
y_pred_lowess = best_est.predict(X_test, return_ci=False)
rmse_lowess = root_mean_squared_error(y_test, y_pred_lowess)
print(f"Default LOWESS RMSE: {rmse_lowess}")

Default LOWESS RMSE: 3.443142382783628


In [8]:
best_par

{'frac': np.float64(0.30000000000000004)}

In [9]:
file_path = "C:/Users/sylwester/PycharmProjects/bblr_testing/scripts/automl_datasets/abalone.csv"
cat_features = ['Sex']  
num_features = ['Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']
target = 'Class_number_of_rings'
test_size = 0.2

X_train, X_test, y_train, y_test = prepare_and_split_data(file_path, cat_features, num_features, target, test_size)

best_est, best_par = optimize_ball_based_regression(X_train, y_train)

bblr_best = best_est
bblr_best.fit(X_train, y_train)
y_pred_best = bblr_best.predict(X_test)
rmse_best = root_mean_squared_error(y_test, y_pred_best)
print(f"Default BBLR RMSE: {rmse_best}")

KeyboardInterrupt: 

In [17]:
import pandas as pd
file_path = "automl_datasets/Brazilian_houses.csv"
df = pd.read_csv(file_path)

In [21]:
# check how many missing values we have in each column
# df.isnull().sum()
df.isna().sum()

city                    0
area                    0
rooms                   0
bathroom                0
parking_spaces          0
floor                   0
animal                  0
furniture               0
hoa_(BRL)               0
rent_amount_(BRL)       0
property_tax_(BRL)      0
fire_insurance_(BRL)    0
total_(BRL)             0
dtype: int64

In [ ]:
file_path = "C:/Users/sylwester/PycharmProjects/bblr_testing/scripts/automl_datasets/house_sales.csv"
cat_features=[],
num_features=['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors','waterfront', 'view', 
                'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 
                'lat', 'long', 'sqft_living15', 'sqft_lot15'],
target='price',
drop_features=['date_year', 'date_month', 'date_day', 'zipcode']
df = pd.read_csv(file_path)

In [10]:
len(num_features)

1

In [13]:
num_features[0]

['bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'lat',
 'long']

In [15]:
df.columns

Index(['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
       'sqft_living15', 'sqft_lot15', 'date_year', 'date_month', 'date_day'],
      dtype='str')

In [23]:
file_path = "C:/Users/sylwester/PycharmProjects/bblr_testing/scripts/automl_datasets/brexitdata.csv"

subset = [
    "c11HouseOutright",
    "c11HouseMortgage",
    "c11HouseholdOnePerson",
    "c11HouseholdMarried",
    "c11CarsNone",
    "c11CarsOne",
    "c11CarsTwo",
    "c11NSSECLowerManager",
    "c11QualNone",
    "c11QualLevel4",
    "c11HealthVeryGood",
    "c11HealthGood",
    "c11DeprivedNone",
    "c11Deprived1",
    'leaveHanretty'
]

df = pd.read_csv(file_path)
df_subset = df[subset]

In [25]:
df_subset.to_csv("automl_datasets/brexit.csv", index=False)

In [3]:
file_path = "C:/Users/sylwester/PycharmProjects/bblr_testing/scripts/automl_datasets/auto_mpg_raw.csv"
df = pd.read_csv(file_path)

# remove rows with missing data
df.replace('?', np.nan, inplace=True)
df = df.dropna()
df.to_csv("automl_datasets/auto_mpg.csv", index=False)